# Lab 06: Root Finding

**Course:** MATH 170 — Introduction to Scientific Computing  
**Term:** Spring 2026

In this lab, you will implement two methods for solving `f(x) = 0`:

- **Bisection** (robust; guaranteed to converge if you have a valid bracket)
- **Newton's method** (usually much faster; can fail)

You will compare their convergence and solve a small loan problem by root finding.

**Instructions:**
1. Implement the functions in the cells below.
2. Run the test cells to verify your code.
3. Copy your implementations into `submission.py` and submit via Git.

In [ ]:
# Setup
import math
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)
plt.rcParams['figure.figsize'] = (8, 4)

## 1) Bisection method

Bisection requires an interval `[a, b]` where `f(a)` and `f(b)` have opposite signs.

At each iteration:
1. Compute the midpoint `m = (a + b) / 2`.
2. Keep the half-interval that still contains a sign change.

Implement:
- `bisection(f, a, b, tol=..., max_iter=...) -> (root, iterations, history)`

`history` should store the midpoints `m` (one per iteration).

In [ ]:
def bisection(f, a, b, tol=1e-10, max_iter=100):
    # Return (root, iterations, history).
    # Raise ValueError if there is no sign change on [a, b].
    # history should store the midpoint from each iteration.
    #
    # TODO: Implement
    return None


In [ ]:
# TEST YOUR CODE

f = lambda x: x**2 - 2

result = bisection(f, 1, 2, tol=1e-6)

if result is None:
    print('❌ Function not implemented yet.')
else:
    # 1) returns (root, iters, history)
    assert isinstance(result, tuple) and len(result) == 3

    # 2) sqrt(2)
    root, iters, history = bisection(f, 1, 2, tol=1e-10)
    assert math.isclose(root, math.sqrt(2), rel_tol=1e-9)

    # 3) iterations reasonable
    assert 10 < iters < 60

    # 4) history length matches
    root2, iters2, history2 = bisection(f, 1, 2, tol=1e-6)
    assert len(history2) == iters2

    # 5) tolerance respected
    g = lambda x: x**3 - x - 2
    root3, iters3, history3 = bisection(g, 1, 2, tol=1e-8)
    assert abs(g(root3)) < 1e-7

    # 6) sin(x) root near pi
    h = lambda x: math.sin(x)
    root4, iters4, history4 = bisection(h, 3, 4, tol=1e-10)
    assert math.isclose(root4, math.pi, rel_tol=1e-9)

    # 7) requires sign change
    p = lambda x: x**2 + 1
    try:
        bisection(p, -1, 1)
        raise AssertionError('Expected ValueError')
    except ValueError:
        pass

    print('✅ Tests passed!')

## 2) Newton's method

Newton's update is:

`x_{n+1} = x_n - f(x_n) / f'(x_n)`

Implement:
- `newton(f, df, x0, tol=..., max_iter=...) -> (root, iterations, history)`

`history` should store the iterates `x_1, x_2, ...`.

In [ ]:
def newton(f, df, x0, tol=1e-10, max_iter=50):
    # Return (root, iterations, history).
    # Stop when abs(f(x)) < tol.
    # Raise ValueError if the derivative is too small.
    # history should store x_1, x_2, ... (one per update).
    #
    # TODO: Implement
    return None


In [ ]:
# TEST YOUR CODE

f = lambda x: x**2 - 2
df = lambda x: 2 * x

result = newton(f, df, 1.5, tol=1e-6)

if result is None:
    print('❌ Function not implemented yet.')
else:
    # 1) returns (root, iters, history)
    assert isinstance(result, tuple) and len(result) == 3

    # 2) sqrt(2)
    root, iters, history = newton(f, df, 1.5, tol=1e-10)
    assert math.isclose(root, math.sqrt(2), rel_tol=1e-9)

    # 3) fast convergence
    assert iters < 10

    # 4) history length matches
    root2, iters2, history2 = newton(f, df, 1.5, tol=1e-6)
    assert len(history2) == iters2

    # 5) cubic root
    g = lambda x: x**3 - x - 2
    dg = lambda x: 3 * x**2 - 1
    root3, iters3, history3 = newton(g, dg, 1.5, tol=1e-10)
    assert abs(g(root3)) < 1e-9

    # 6) derivative too small
    h = lambda x: x**3 - 1
    dh = lambda x: 3 * x**2
    try:
        newton(h, dh, 0.0)
        raise AssertionError('Expected ValueError')
    except ValueError:
        pass

    print('✅ Tests passed!')

## 3) Convergence comparison

Compare bisection vs Newton on `f(x) = x**2 - 2`.

Make a semilog plot of the error vs iteration using the histories returned by your functions.

In [ ]:
# Compare convergence on sqrt(2)

f = lambda x: x**2 - 2
df = lambda x: 2 * x
true_root = math.sqrt(2)

b_root, b_iters, b_hist = bisection(f, 1.0, 2.0, tol=1e-12)
n_root, n_iters, n_hist = newton(f, df, 1.5, tol=1e-12)

b_errors = [abs(m - true_root) for m in b_hist]
n_errors = [abs(x - true_root) for x in n_hist]

plt.semilogy(range(1, len(b_errors) + 1), b_errors, 'b.-', label=f'Bisection ({b_iters} iters)')
plt.semilogy(range(1, len(n_errors) + 1), n_errors, 'r.-', label=f'Newton ({n_iters} iters)')
plt.xlabel('Iteration')
plt.ylabel('Error')
plt.title('Convergence on x^2 - 2 = 0')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 4) Application: implied interest rate

We model a fixed-payment loan with monthly compounding.

- `rate` is the annual interest rate `r`.
- monthly rate is `r_m = r / 12`.
- `principal` is the initial balance `B_0`.
- `payment` is the monthly payment `p`.

Update rule:

`B_{k+1} = B_k * (1 + r_m) - p`

Define the residual as the remaining balance after `n_months` payments.

Implement:
- `loan_payment_residual(rate, principal, payment, n_months)`
- `find_interest_rate(principal, payment, n_months)` (use your `bisection`)

In [ ]:
def loan_payment_residual(rate, principal, payment, n_months):
    # Return the remaining balance after n_months payments.
    # rate is an annual rate (e.g. 0.12 for 12%).
    # Special case: rate == 0.
    #
    # TODO: Implement
    return None


In [ ]:
def find_interest_rate(principal, payment, n_months):
    # Find the annual interest rate that makes the residual ~ 0.
    # Use bisection on a reasonable interval (e.g. [0.0, 0.50]).
    #
    # TODO: Implement
    return None


In [ ]:
# TEST YOUR CODE

# Loan residual tests
residual = loan_payment_residual(0.0, 1200, 100, 12)

if residual is None:
    print('❌ Function not implemented yet: loan_payment_residual')
else:
    assert abs(residual) < 1e-6

    residual = loan_payment_residual(0.12, 1000, 50, 12)
    assert residual > 0

    residual = loan_payment_residual(0.12, 1000, 200, 12)
    assert residual < 0

    print('✅ loan_payment_residual passed!')

    # Interest rate tests
    rate = find_interest_rate(10000, 879.16, 12)

    if rate is None:
        print('❌ Function not implemented yet: find_interest_rate')
    else:
        assert math.isclose(rate, 0.10, rel_tol=0.01)

        rate2 = find_interest_rate(1000, 100, 12)
        assert 0 <= rate2 <= 1

        rate3 = find_interest_rate(1200, 100.5, 12)
        assert rate3 < 0.05

        print('✅ find_interest_rate passed!')